In [19]:
import ast

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

PRIOR_FW_WIN = 5  # mean of last N fw values (excluding current) used as picking target

hr_accel = pd.read_csv('sleep/hr_accel_aniket.csv')


def _parse_peaks(v):
    if isinstance(v, list):
        return v
    if isinstance(v, float) and np.isnan(v):
        return []
    return ast.literal_eval(v) if isinstance(v, str) and v.strip() else []


def _summary(name, e):
    abs_e = np.abs(e)
    if not len(e):
        print(f"{name}: n=0")
        return
    print(f"{name}: n={len(e)}, mean={e.mean():.2f}, std={e.std():.2f}, "
          f"|err| median={np.median(abs_e):.2f}, |err| p95={np.percentile(abs_e, 95):.2f}")


def _closest_to(target, peaks):
    if not peaks or not np.isfinite(target) or target <= 0:
        return np.nan
    arr = np.asarray(peaks, dtype=float)
    return float(arr[np.argmin(np.abs(arr - target))])

fw_df = hr_accel.copy()
fw_df['fused_mag_peaks'] = fw_df['fused_mag_peaks'].apply(_parse_peaks)
fw_df['fused_pca_peaks'] = fw_df['fused_pca_peaks'].apply(_parse_peaks)
fw_df = fw_df[fw_df['firmware_hr'].notna() & (fw_df['firmware_hr'] > 0)].copy()
fw_df['window_start'] = pd.to_datetime(fw_df['window_start'])
fw_df = fw_df.sort_values('window_start').reset_index(drop=True)

# Picking target: mean of last PRIOR_FW_WIN fw values, NOT including current.
# Fall back to current fw when fewer than PRIOR_FW_WIN priors are available.
prior_target = (fw_df['firmware_hr']
                .rolling(PRIOR_FW_WIN, min_periods=1)
                .mean()
                .shift(1))
prior_target = prior_target.fillna(fw_df['firmware_hr'])
fw_df['pick_target'] = prior_target

fw_df['err_green_fw'] = fw_df['green'] - fw_df['firmware_hr']
fw_df['err_mag_fw'] = [
    _closest_to(t, p) - fw
    for t, fw, p in zip(fw_df['pick_target'], fw_df['firmware_hr'], fw_df['fused_mag_peaks'])
]
fw_df['err_pca_fw'] = [
    _closest_to(t, p) - fw
    for t, fw, p in zip(fw_df['pick_target'], fw_df['firmware_hr'], fw_df['fused_pca_peaks'])
]

mag_abs = fw_df['err_mag_fw'].abs()
pca_abs = fw_df['err_pca_fw'].abs()
use_mag = mag_abs <= pca_abs
fw_df['err_best_fw'] = np.where(use_mag, fw_df['err_mag_fw'], fw_df['err_pca_fw'])
fw_df['best_src'] = np.where(use_mag, 'mag', 'pca')

err_green_fw = fw_df['err_green_fw'].dropna().to_numpy()
err_mag_fw = fw_df['err_mag_fw'].dropna().to_numpy()
err_pca_fw = fw_df['err_pca_fw'].dropna().to_numpy()
err_best_fw = fw_df['err_best_fw'].dropna().to_numpy()

fw_g = fw_df.loc[fw_df['err_green_fw'].notna(), 'firmware_hr'].to_numpy()
fw_m = fw_df.loc[fw_df['err_mag_fw'].notna(), 'firmware_hr'].to_numpy()
fw_p = fw_df.loc[fw_df['err_pca_fw'].notna(), 'firmware_hr'].to_numpy()
fw_b = fw_df.loc[fw_df['err_best_fw'].notna(), 'firmware_hr'].to_numpy()

print(f"firmware_hr available on {len(fw_df)}/{len(hr_accel)} windows")
print(f"pick target: mean of last {PRIOR_FW_WIN} fw values (excl. current); error vs current fw")
print(f"best source split: mag={int(use_mag.sum())}, pca={int((~use_mag).sum())}")
_summary('green             vs fw', err_green_fw)
_summary('fused_mag         vs fw', err_mag_fw)
_summary('fused_pca         vs fw', err_pca_fw)
_summary('best(mag,pca)     vs fw', err_best_fw)

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=(
        'green - fw (bpm)',
        'closest fused_mag - fw (bpm)',
        'closest fused_pca - fw (bpm)',
        'best(mag,pca) - fw (bpm)',
        'fw vs green',
        'fw vs closest fused_mag',
        'fw vs closest fused_pca',
        'fw vs best(mag,pca)',
    ),
)

bin_size = 0.5
all_err = np.concatenate([err_green_fw, err_mag_fw, err_pca_fw, err_best_fw])
xbins = dict(start=float(np.floor(all_err.min())),
             end=float(np.ceil(all_err.max())),
             size=bin_size)
fig.add_trace(go.Histogram(x=err_green_fw, xbins=xbins, marker_color='#2ca02c', autobinx=False),
              row=1, col=1)
fig.add_trace(go.Histogram(x=err_mag_fw, xbins=xbins, marker_color='#e377c2', autobinx=False),
              row=1, col=2)
fig.add_trace(go.Histogram(x=err_pca_fw, xbins=xbins, marker_color='#8c564b', autobinx=False),
              row=1, col=3)
fig.add_trace(go.Histogram(x=err_best_fw, xbins=xbins, marker_color='#1f77b4', autobinx=False),
              row=1, col=4)

lo, hi = 40, 180
fig.add_trace(go.Scatter(x=fw_g, y=fw_g + err_green_fw, mode='markers',
                         marker=dict(color='#2ca02c', size=4, opacity=0.5)),
              row=2, col=1)
fig.add_trace(go.Scatter(x=fw_m, y=fw_m + err_mag_fw, mode='markers',
                         marker=dict(color='#e377c2', size=4, opacity=0.5)),
              row=2, col=2)
fig.add_trace(go.Scatter(x=fw_p, y=fw_p + err_pca_fw, mode='markers',
                         marker=dict(color='#8c564b', size=4, opacity=0.5)),
              row=2, col=3)
fig.add_trace(go.Scatter(x=fw_b, y=fw_b + err_best_fw, mode='markers',
                         marker=dict(color='#1f77b4', size=4, opacity=0.5)),
              row=2, col=4)
for col in (1, 2, 3, 4):
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines',
                             line=dict(color='gray', dash='dash')),
                  row=2, col=col)

for col in (1, 2, 3, 4):
    fig.update_xaxes(title_text='error (bpm)', row=1, col=col)
    fig.update_xaxes(title_text='firmware_hr (bpm)', row=2, col=col)
fig.update_yaxes(title_text='count', row=1, col=1)
fig.update_yaxes(title_text='estimate (bpm)', row=2, col=1)
fig.update_layout(height=720, showlegend=False,
                  title_text='Error vs firmware_hr — green, fused_mag, fused_pca, best')
fig.show()

firmware_hr available on 1616/1616 windows
pick target: mean of last 5 fw values (excl. current); error vs current fw
best source split: mag=915, pca=701
green             vs fw: n=1616, mean=1.47, std=3.51, |err| median=2.10, |err| p95=6.63
fused_mag         vs fw: n=1611, mean=-2.18, std=9.63, |err| median=5.93, |err| p95=20.23
fused_pca         vs fw: n=1616, mean=-1.09, std=9.49, |err| median=5.47, |err| p95=19.43
best(mag,pca)     vs fw: n=1616, mean=-0.57, std=6.04, |err| median=3.30, |err| p95=12.10


In [20]:
# Windows where green tracks fw closely but best(mag,pca) diverges.
GREEN_OK_BPM = 1.0
BEST_BAD_BPM = 2.0

mask = (fw_df['err_green_fw'].abs() < GREEN_OK_BPM) & (fw_df['err_best_fw'].abs() > BEST_BAD_BPM)
bad = fw_df.loc[mask].copy()
bad = bad.sort_values('window_start')

print(f"{len(bad)} windows where |green-fw| < {GREEN_OK_BPM} and |best-fw| > {BEST_BAD_BPM}")
cols = ['email', 'sleep_date', 'window_start', 'firmware_hr', 'green',
        'err_green_fw', 'err_mag_fw', 'err_pca_fw', 'err_best_fw', 'best_src',
        'fused_mag_peaks', 'fused_pca_peaks']
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', None, 'display.width', 240):
    print(bad[cols].to_string(index=False))

225 windows where |green-fw| < 1.0 and |best-fw| > 2.0
                 email sleep_date                     window_start  firmware_hr     green  err_green_fw  err_mag_fw  err_pca_fw  err_best_fw best_src                                                                                                      fused_mag_peaks                                                                     fused_pca_peaks
aniket.jana@temple.com 2026-05-08 2026-05-07 23:48:45.018000+05:30    92.864017 92.929688      0.065671   11.135983  -10.758754   -10.758754      pca                                                                       [156.0, 103.99999999999999, 78.0, 60.0, 48.75]                   [82.10526315789473, 43.333333333333336, 55.71428571428571, 156.0]
aniket.jana@temple.com 2026-05-08 2026-05-07 23:52:15.018000+05:30    90.866388 91.406250      0.539862  -16.580674   -8.761125    -8.761125      pca                                                           [111.42857142857142, 74.28571428571

In [21]:
# Time series across the night: firmware_hr, green peak, best(mag,pca).
# Raw points shown faintly. firmware_hr smoothed with rolling median;
# green and best smoothed with rolling EMA.
SMOOTH_WIN = 15      # window for firmware_hr median; also EMA span for green/best
TIME_FROM = None     # e.g. '2026-05-15 23:30' — None = no lower bound
TIME_TO = None       # e.g. '2026-05-16 03:00' — None = no upper bound

ts_df = fw_df.copy()
ts_df['window_start'] = pd.to_datetime(ts_df['window_start'])
ts_df['best_hr'] = ts_df['firmware_hr'] + ts_df['err_best_fw']
ts_df = ts_df.sort_values('window_start').reset_index(drop=True)

if TIME_FROM is not None or TIME_TO is not None:
    tz = ts_df['window_start'].dt.tz
    lo = pd.Timestamp(TIME_FROM) if TIME_FROM is not None else None
    hi = pd.Timestamp(TIME_TO) if TIME_TO is not None else None
    if lo is not None and lo.tz is None and tz is not None:
        lo = lo.tz_localize(tz)
    if hi is not None and hi.tz is None and tz is not None:
        hi = hi.tz_localize(tz)
    if lo is not None:
        ts_df = ts_df[ts_df['window_start'] >= lo]
    if hi is not None:
        ts_df = ts_df[ts_df['window_start'] <= hi]
    ts_df = ts_df.reset_index(drop=True)

if len(ts_df):
    print(f"plotting {len(ts_df)} windows "
          f"({ts_df['window_start'].min()} → {ts_df['window_start'].max()})")
else:
    print("no windows in selected time range")

fw_smooth = ts_df['firmware_hr'].rolling(
    SMOOTH_WIN, center=True, min_periods=max(3, SMOOTH_WIN // 3)
).median()
green_smooth = ts_df['green'].ewm(span=SMOOTH_WIN, adjust=False).mean()
best_smooth = ts_df['best_hr'].ewm(span=SMOOTH_WIN, adjust=False).mean()

series = [
    ('firmware_hr',   'firmware_hr',   '#1f77b4', fw_smooth),
    ('green',         'green peak',    '#2ca02c', green_smooth),
    ('best_hr',       'best(mag,pca)', '#e377c2', best_smooth),
]

fig = go.Figure()
for col, label, color, smooth in series:
    fig.add_trace(go.Scatter(x=ts_df['window_start'], y=ts_df[col],
                             mode='markers', name=f'{label} (raw)',
                             marker=dict(size=3, color=color, opacity=0.25),
                             showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=ts_df['window_start'], y=smooth,
                             mode='lines', name=label,
                             line=dict(color=color, width=2)))

fig.update_xaxes(title_text='time')
fig.update_yaxes(title_text='HR (bpm)')
fig.update_layout(height=500,
                  title_text=f'HR estimates across the night (firmware: median, green/best: EMA, win={SMOOTH_WIN})',
                  hovermode='x unified')
fig.show()

plotting 1616 windows (2026-05-07 23:45:15.018000+05:30 → 2026-05-08 07:44:03.018000+05:30)


---

In [10]:
# --- Respiration: same analysis vs green peak (no firmware truth) ---------
RESP_CSV = 'sleep/resp_accel_sanchit.csv'    # output of respiration_picker.pick_all_dense_windows
PRIOR_GREEN_WIN = 5                     # mean of last N green values (excl. current)


def _parse_peaks_resp(v):
    if isinstance(v, list):
        return v
    if isinstance(v, float) and np.isnan(v):
        return []
    return ast.literal_eval(v) if isinstance(v, str) and v.strip() else []


resp = pd.read_csv(RESP_CSV)
resp['fused_mag_peaks'] = resp['fused_mag_peaks'].apply(_parse_peaks_resp)
resp['fused_pca_peaks'] = resp['fused_pca_peaks'].apply(_parse_peaks_resp)
resp = resp[resp['green'].notna() & (resp['green'] > 0)].copy()
resp['window_start'] = pd.to_datetime(resp['window_start'])
resp = resp.sort_values('window_start').reset_index(drop=True)

# Picking target: mean of last PRIOR_GREEN_WIN green values, NOT including current.
# Fall back to current green when fewer priors are available.
prior_target_g = (resp['green']
                  .rolling(PRIOR_GREEN_WIN, min_periods=1)
                  .mean()
                  .shift(1))
prior_target_g = prior_target_g.fillna(resp['green'])
resp['pick_target'] = prior_target_g

resp['err_mag_g'] = [
    _closest_to(t, p) - g
    for t, g, p in zip(resp['pick_target'], resp['green'], resp['fused_mag_peaks'])
]
resp['err_pca_g'] = [
    _closest_to(t, p) - g
    for t, g, p in zip(resp['pick_target'], resp['green'], resp['fused_pca_peaks'])
]

mag_abs = resp['err_mag_g'].abs()
pca_abs = resp['err_pca_g'].abs()
use_mag_r = mag_abs <= pca_abs
resp['err_best_g'] = np.where(use_mag_r, resp['err_mag_g'], resp['err_pca_g'])
resp['best_src'] = np.where(use_mag_r, 'mag', 'pca')

# get_resp (smart-fusion) error vs green; treat 0 as missing (early-exit signal).
sf = pd.to_numeric(resp.get('resp_smartfusion'), errors='coerce')
resp['resp_smartfusion'] = sf.where(sf > 0)
resp['err_sf_g'] = resp['resp_smartfusion'] - resp['green']

err_mag_g = resp['err_mag_g'].dropna().to_numpy()
err_pca_g = resp['err_pca_g'].dropna().to_numpy()
err_best_g = resp['err_best_g'].dropna().to_numpy()
err_sf_g = resp['err_sf_g'].dropna().to_numpy()

g_m = resp.loc[resp['err_mag_g'].notna(), 'green'].to_numpy()
g_p = resp.loc[resp['err_pca_g'].notna(), 'green'].to_numpy()
g_b = resp.loc[resp['err_best_g'].notna(), 'green'].to_numpy()
g_s = resp.loc[resp['err_sf_g'].notna(), 'green'].to_numpy()

print(f"resp rows: {len(resp)}  (smartfusion valid: {int(resp['resp_smartfusion'].notna().sum())})")
print(f"pick target: mean of last {PRIOR_GREEN_WIN} green values (excl. current); error vs current green")
print(f"best source split: mag={int(use_mag_r.sum())}, pca={int((~use_mag_r).sum())}")
_summary('fused_mag         vs green', err_mag_g)
_summary('fused_pca         vs green', err_pca_g)
_summary('best(mag,pca)     vs green', err_best_g)
_summary('get_resp (smartf) vs green', err_sf_g)

fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=(
        'closest fused_mag - green (brpm)',
        'closest fused_pca - green (brpm)',
        'best(mag,pca) - green (brpm)',
        'get_resp - green (brpm)',
        'green vs closest fused_mag',
        'green vs closest fused_pca',
        'green vs best(mag,pca)',
        'green vs get_resp',
    ),
)

bin_size = 0.25
all_err_arrays = [a for a in (err_mag_g, err_pca_g, err_best_g, err_sf_g) if a.size]
all_err = np.concatenate(all_err_arrays) if all_err_arrays else np.array([0.0])
xbins = dict(start=float(np.floor(all_err.min())),
             end=float(np.ceil(all_err.max())),
             size=bin_size)
fig.add_trace(go.Histogram(x=err_mag_g, xbins=xbins, marker_color='#e377c2', autobinx=False),
              row=1, col=1)
fig.add_trace(go.Histogram(x=err_pca_g, xbins=xbins, marker_color='#8c564b', autobinx=False),
              row=1, col=2)
fig.add_trace(go.Histogram(x=err_best_g, xbins=xbins, marker_color='#1f77b4', autobinx=False),
              row=1, col=3)
fig.add_trace(go.Histogram(x=err_sf_g, xbins=xbins, marker_color='#ff7f0e', autobinx=False),
              row=1, col=4)

lo, hi = 6, 30
fig.add_trace(go.Scatter(x=g_m, y=g_m + err_mag_g, mode='markers',
                         marker=dict(color='#e377c2', size=4, opacity=0.5)),
              row=2, col=1)
fig.add_trace(go.Scatter(x=g_p, y=g_p + err_pca_g, mode='markers',
                         marker=dict(color='#8c564b', size=4, opacity=0.5)),
              row=2, col=2)
fig.add_trace(go.Scatter(x=g_b, y=g_b + err_best_g, mode='markers',
                         marker=dict(color='#1f77b4', size=4, opacity=0.5)),
              row=2, col=3)
fig.add_trace(go.Scatter(x=g_s, y=g_s + err_sf_g, mode='markers',
                         marker=dict(color='#ff7f0e', size=4, opacity=0.5)),
              row=2, col=4)
for col in (1, 2, 3, 4):
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode='lines',
                             line=dict(color='gray', dash='dash')),
                  row=2, col=col)

for col in (1, 2, 3, 4):
    fig.update_xaxes(title_text='error (brpm)', row=1, col=col)
    fig.update_xaxes(title_text='green (brpm)', row=2, col=col)
fig.update_yaxes(title_text='count', row=1, col=1)
fig.update_yaxes(title_text='estimate (brpm)', row=2, col=1)
fig.update_layout(height=720, showlegend=False,
                  title_text='Respiration error vs green — fused_mag, fused_pca, best, get_resp')
fig.show()

resp rows: 474  (smartfusion valid: 248)
pick target: mean of last 5 green values (excl. current); error vs current green
best source split: mag=287, pca=187
fused_mag         vs green: n=474, mean=-0.03, std=1.77, |err| median=0.86, |err| p95=3.78
fused_pca         vs green: n=473, mean=-0.01, std=1.98, |err| median=1.06, |err| p95=4.48
best(mag,pca)     vs green: n=473, mean=-0.08, std=1.32, |err| median=0.59, |err| p95=2.32
get_resp (smartf) vs green: n=248, mean=1.64, std=2.97, |err| median=1.89, |err| p95=6.47


In [11]:
# Respiration windows where best(mag,pca) diverges most from green peak.
RESP_BEST_BAD_BRPM = 2.0

mask_r = resp['err_best_g'].abs() > RESP_BEST_BAD_BRPM
bad_r = resp.loc[mask_r].copy()
bad_r = bad_r.sort_values('window_start')

print(f"{len(bad_r)} resp windows where |best - green| > {RESP_BEST_BAD_BRPM}")
cols_r = ['email', 'sleep_date', 'window_start', 'green',
          'err_mag_g', 'err_pca_g', 'err_best_g', 'best_src',
          'fused_mag_peaks', 'fused_pca_peaks']
with pd.option_context('display.max_rows', 200, 'display.max_colwidth', None, 'display.width', 240):
    print(bad_r[cols_r].to_string(index=False))

36 resp windows where |best - green| > 2.0
             email sleep_date                     window_start     green  err_mag_g  err_pca_g  err_best_g best_src                                                                    fused_mag_peaks                                                                fused_pca_peaks
sanchit@temple.com 2026-05-16 2026-05-15 23:02:57.018000+05:30 15.234375  -7.910431  -7.698143   -7.698143      pca                          [15.445544554455443, 7.32394366197183, 9.176470588235293]                                         [7.536231884057971, 14.71698113207547]
sanchit@temple.com 2026-05-16 2026-05-15 23:45:17.018000+05:30 13.710938  -6.521997  -6.928329   -6.521997      mag                          [9.12280701754386, 17.727272727272727, 7.188940092165899]                                        [13.805309734513273, 6.782608695652174]
sanchit@temple.com 2026-05-16 2026-05-15 23:48:01.018000+05:30 12.187500  -3.845254  -6.045768   -3.845254      mag      [1

In [12]:
# Respiration time series across the night: green peak, best(mag,pca), get_resp.
# Raw points shown faintly. green smoothed with rolling median;
# best and get_resp smoothed with rolling EMA.
SMOOTH_WIN_R = 15      # window for green median; also EMA span for best / get_resp
TIME_FROM_R = None     # e.g. '2026-05-15 23:30' — None = no lower bound
TIME_TO_R = None       # e.g. '2026-05-16 03:00' — None = no upper bound

ts_r = resp.copy()
ts_r['window_start'] = pd.to_datetime(ts_r['window_start'])
ts_r['best_brpm'] = ts_r['green'] + ts_r['err_best_g']
ts_r = ts_r.sort_values('window_start').reset_index(drop=True)

if TIME_FROM_R is not None or TIME_TO_R is not None:
    tz = ts_r['window_start'].dt.tz
    lo_t = pd.Timestamp(TIME_FROM_R) if TIME_FROM_R is not None else None
    hi_t = pd.Timestamp(TIME_TO_R) if TIME_TO_R is not None else None
    if lo_t is not None and lo_t.tz is None and tz is not None:
        lo_t = lo_t.tz_localize(tz)
    if hi_t is not None and hi_t.tz is None and tz is not None:
        hi_t = hi_t.tz_localize(tz)
    if lo_t is not None:
        ts_r = ts_r[ts_r['window_start'] >= lo_t]
    if hi_t is not None:
        ts_r = ts_r[ts_r['window_start'] <= hi_t]
    ts_r = ts_r.reset_index(drop=True)

if len(ts_r):
    print(f"plotting {len(ts_r)} resp windows "
          f"({ts_r['window_start'].min()} → {ts_r['window_start'].max()})  "
          f"smartfusion valid: {int(ts_r['resp_smartfusion'].notna().sum())}")
else:
    print("no resp windows in selected time range")

green_smooth_r = ts_r['green'].rolling(
    SMOOTH_WIN_R, center=True, min_periods=max(3, SMOOTH_WIN_R // 3)
).median()
best_smooth_r = ts_r['best_brpm'].ewm(span=SMOOTH_WIN_R, adjust=False).mean()
sf_smooth_r = ts_r['resp_smartfusion'].ewm(span=SMOOTH_WIN_R, adjust=False).mean()

series_r = [
    ('green',            'green peak',    '#2ca02c', green_smooth_r),
    ('best_brpm',        'best(mag,pca)', '#e377c2', best_smooth_r),
    ('resp_smartfusion', 'get_resp',      '#ff7f0e', sf_smooth_r),
]

fig = go.Figure()
for col, label, color, smooth in series_r:
    fig.add_trace(go.Scatter(x=ts_r['window_start'], y=ts_r[col],
                             mode='markers', name=f'{label} (raw)',
                             marker=dict(size=3, color=color, opacity=0.25),
                             showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=ts_r['window_start'], y=smooth,
                             mode='lines', name=label,
                             line=dict(color=color, width=2)))

fig.update_xaxes(title_text='time')
fig.update_yaxes(title_text='resp rate (brpm)')
fig.update_layout(height=500,
                  title_text=f'Respiration estimates across the night (green: median, best/get_resp: EMA, win={SMOOTH_WIN_R})',
                  hovermode='x unified')
fig.show()

plotting 474 resp windows (2026-05-15 22:35:15.018000+05:30 → 2026-05-16 08:08:35.018000+05:30)  smartfusion valid: 248
